In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# הצגת חזותית של תזמון מעגלים

<Accordion>
<AccordionItem title="גרסאות חבילות">

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלה או חדשות יותר.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

בעוד ש[כלי ציור הציר הזמני](/guides/visualize-circuit-timing) המובנה ב-Qiskit שימושי למעגלים סטטיים, ייתכן שהוא לא ישקף במדויק את התזמון של [מעגלים דינמיים](/guides/classical-feedforward-and-control-flow) בגלל פעולות משתמעות כמו broadcasting וקביעת ענפים. כחלק מתמיכת מעגלים דינמיים, Qiskit Runtime מחזיר את מידע תזמון המעגל המדויק בתוצאות העבודה כאשר מבקשים זאת.

> **Note:** - זוהי פונקציה ניסיונית. היא נמצאת במצב שחרור תצוגה מקדימה ולכן עשויה להשתנות.
> - פונקציה זו חלה רק על עבודות Qiskit Runtime Sampler.
> - למרות שזמן המעגל הכולל מוחזר ב-metadata של "compilation", זה **אינו** הזמן המשמש לחיוב (זמן QPU).

### אפשר שליפת נתוני תזמון
כדי לאפשר שליפת נתוני תזמון, הגדר את הדגל הניסיוני `scheduler_timing` ל-`True` בעת הרצת עבודת ה-primitive.

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### גישה לנתוני תזמון המעגל
כאשר מבקשים זאת, נתוני תזמון המעגל לכל PUB מוחזרים ב-metadata של תוצאת העבודה, תחת `["compilation"]["scheduler_timing"]["timing"]`. שדה זה מכיל את מידע התזמון הגולמי. כדי להציג את מידע התזמון, השתמש בכלי ההצגה החזותית המובנה, כמתואר בחלק [הצגת חזותית של התזמונים](#visualize-timings).

השתמש בקוד הבא כדי לגשת לנתוני תזמון המעגל עבור ה-PUB הראשון:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### הבנת נתוני התזמון הגולמיים
בעוד שהצגת נתוני תזמון המעגל באמצעות שיטת `draw_circuit_schedule_timing` היא מקרה השימוש הנפוץ ביותר, ייתכן שיהיה שימושי להבין את מבנה נתוני התזמון הגולמיים שמוחזרים. זה יכול לעזור לך, לדוגמה, לחלץ מידע באופן תכנותי.

נתוני התזמון שמוחזרים ב-`["compilation"]["scheduler_timing"]["timing"]` הם רשימה של מחרוזות. כל מחרוזת מייצגת הוראה בודדת על ערוץ מסוים ומופרדת בפסיקים לסוגי הנתונים הבאים:

- `Branch` - קובע האם ההוראה נמצאת בזרימת בקרה (then / else) או בענף ראשי.
- `Instruction` - ה-Gate וה-Qubit לפעול עליו.
- `Channel` - הערוץ שמוקצה עם ההוראה. יכול להיות אחד מהבאים:
      - `Qubit x` - ערוץ ה-drive עבור Qubit _x_.
      - `AWGRx_y` (arbitrary waveform generator readout) - משמש לערוצי קריאה לתקשורת בעת מדידת Qubits. הארגומנטים _x_ ו-_y_ מתאימים למזהה מכשיר הקריאה ומספר ה-Qubit, בהתאמה.
- `T0` - זמן ההתחלה של ההוראה בתוך לוח הזמנים המלא
- `Duration` - משך ההוראה, ביחידות של שניות _dt_, כאשר 1 dt = 1 מחזור תזמון. ניתן למצוא את ערך ה-`dt` של backend באמצעות [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - סוג פעולת ה-pulse בשימוש.

דוגמה:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### הצגת חזותית של התזמונים
עם `qiskit-ibm-runtime` v0.43.0 ומעלה, ניתן להציג חזותית תזמוני מעגלים. כדי להציג חזותית את התזמונים, תחילה צריך להמיר את ה-metadata של התוצאה ל-`fig` באמצעות [שיטת `draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26). שיטה זו מחזירה `plotly` figure, שאפשר להציג ישירות, לשמור לקובץ, או שניהם. למידע נוסף על פקודות ה-`plotly` לשימוש, ראו [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) ו-[`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html).

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![מעבר עם העכבר על הפלט מציג מידע כמו התחלה, סיום ומשך.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'דוגמה לאיור שנוצר')

#### הבנת האיור שנוצר
התמונה של פלט נתוני תזמון המעגל שנוצרה על ידי `draw_circuit_schedule_timing` מעבירה את המידע הבא:

- ציר ה-X הוא זמן ביחידות של שניות _dt_, כאשר 1 dt = 1 מחזור תזמון. ניתן למצוא את ערך ה-`dt` של backend באמצעות [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- ציר ה-Y הוא הערוץ (חשוב על ערוצים כמכשירים שפולטים pulses).
    - `Receive channel` - הערוץ היחיד שאינו מכשיר בפני עצמו. זוהי הוראה שמתנגנת על כל הערוצים שהם חלק מנוהל תקשורת עם ה-hub באותו הזמן.
    - `Qubit x` - ערוץ ה-drive עבור Qubit x.
    - `AWGRx_y` (arbitrary waveform generator readout) - משמש לערוצי קריאה לתקשורת בעת מדידת Qubits. הארגומנטים _x_ ו-_y_ מתאימים למזהה מכשיר הקריאה ומספר ה-Qubit, בהתאמה.
    - `Hub` - שולט ב-broadcasting.

בנוסף, לכל הוראה יש הפורמט *X_Y*, כאשר *X* הוא שם ההוראה ו-*Y* הוא סוג ה-pulse. סוג `play` מיישם pulses בקרה, ו-`capture` מקליט את מצב ה-Qubit. ניתן גם לרחף עם העכבר מעל כל הוראה לקבלת פרטים נוספים. לדוגמה, האיור הקודם מציג pulse בקרה עבור שער ה-X המיושם על Qubit 10 ב-1161 dt.

### דוגמה מקצה לקצה
דוגמה זו מראה לך כיצד לאפשר את האפשרות, לקבל את מידע תזמון המעגל מה-metadata, ולהציג אותו כתמונה.

ראשית, הגדר את הסביבה, הגדר את המעגלים והמר אותם למעגלי ISA, והגדר והרץ את העבודות.

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

לאחר מכן, קבל את תזמון לוח הזמנים של המעגל: